# Source Code

This file details the code used to perform the analysis and generate the graphs. There will be step-by-step descriptions for each cell.

# Part 1: Analysis with the latest FAFB snapshot (v783)

## 1.1 Load edge list into a directed graph

The FAFB edge list was downloaded from [FlyWire](https://codex.flywire.ai/api/download).

In [20]:
import pandas as pd
import networkx as nx
from json import load

edge_list = pd.read_csv("connections_princeton.csv")
edge_list = (
    edge_list
    .groupby(['pre_root_id', 'post_root_id', 'nt_type'], as_index=False)['syn_count']
    .sum()
)
# edge_list_no_neuropil = (
#     edge_list
#     .groupby(['pre_root_id', 'post_root_id', 'nt_type'], as_index=False)['syn_count']
#     .sum()
# )
# print(edge_list.groupby(['pre_root_id','post_root_id'])['syn_count'].sum().sum())  # true total
# print(edge_list_no_neuropil['syn_count'].sum())  # after correct collapse

# dropped = edge_list.drop(columns=['neuropil']).drop_duplicates()
# print(dropped['syn_count'].sum())          # drop_duplicates total
# print(edge_list_no_neuropil['syn_count'].sum())  # groupby.sum() total, known-correct
# print(dropped['syn_count'].sum() - edge_list_no_neuropil['syn_count'].sum())
neurons = None

with open("data/groups.json") as f:
    neurons = load(f)

G = nx.from_pandas_edgelist(edge_list, source="pre_root_id", target="post_root_id", create_using=nx.DiGraph)
IR94E = set(neurons["783"]["Ir94e"])
OVIDN = set(neurons["783"]["OviDN"])

## 1.2 Find shortest paths

This cell iterates through each Ir94e, then through each OviDN, finding the shortest path between each Ir94e to each OviDN.

`nx.algorithms.all_shortest_paths` will return multiple shortest paths if they are identical in length. For each discovered path, we add the edges to a running list called `path_edges`. We also save the interneurons to a set.

Note: this performs a shortest paths analysis across the entire connectome for 90 iterations (18 Ir94e x 5 OviDN), so this cell can run for a while.

In [21]:
INTERNEURONS = set()
path_edges = []

for ir94e in neurons["783"]["Ir94e"]:
    for ovidn in neurons["783"]["OviDN"]:
        print(f"finding pathways between {ir94e} -> {ovidn}...", end="")
        paths: Generator[list, None, None] = nx.algorithms.all_shortest_paths(G, ir94e, ovidn)
        print("done")
        for path in paths:
            if len(path) > 4: continue # only 3 hops/4 neurons or less are considered as per the original paper's methods
            path_edges.extend(zip(path, path[1:])) # zip(path, path[1:]) is a neat shorthand of generating edges from the list of nodes in the path
            INTERNEURONS.update(set(path[1:-1])) # each path starts with Ir94e and ends with OviDN, so the interneurons are the nodes in between

path_edges_df = pd.DataFrame(path_edges, columns=["pre_root_id", "post_root_id"])
print(f"interneurons discovered: {len(INTERNEURONS)}")
print(f"unique path edges discovered: {len(path_edges_df.drop_duplicates())}")

finding pathways between 720575940621375231 -> 720575940632512156...done
finding pathways between 720575940621375231 -> 720575940620625880...done
finding pathways between 720575940621375231 -> 720575940621257340...done
finding pathways between 720575940621375231 -> 720575940627921182...done
finding pathways between 720575940621375231 -> 720575940642312136...done
finding pathways between 720575940638218173 -> 720575940632512156...done
finding pathways between 720575940638218173 -> 720575940620625880...done
finding pathways between 720575940638218173 -> 720575940621257340...done
finding pathways between 720575940638218173 -> 720575940627921182...done
finding pathways between 720575940638218173 -> 720575940642312136...done
finding pathways between 720575940626016017 -> 720575940632512156...done
finding pathways between 720575940626016017 -> 720575940620625880...done
finding pathways between 720575940626016017 -> 720575940621257340...done
finding pathways between 720575940626016017 -> 7205

## 1.3 Filter main edge list

Next, we filter the main edge list to capture edges involved in the circuit with their respective weights. This is done with a simple pandas merge on the source/target columns. The results are saved to a separate file.

In [22]:
circuit_edges = edge_list.merge(
    path_edges_df.drop_duplicates(),
    on=["pre_root_id", "post_root_id"],
    how="inner",
)

print(circuit_edges.head())
print(circuit_edges.shape)
circuit_edges.to_csv("filtered_edge_list.csv", index=False)

          pre_root_id        post_root_id nt_type  syn_count
0  720575940604395436  720575940620625880     ACH          7
1  720575940604395436  720575940621257340     ACH         11
2  720575940604395436  720575940642312136     ACH          5
3  720575940604891360  720575940629013199    GABA          6
4  720575940605040300  720575940642312136    GLUT         11
(302, 4)


## 1.4 Pool connectivity from the neuron level to the group level

The above edge list shows how individual neurons are connected within the graph. However, we are interested in the connectibity between *groups* of neurons.

This requires knowing what neuron IDs fall into which groups. For the sake of recreating the figure, I pulled this information directly from the paper's supplemental information (with some IDs being updated since the paper's publishing).

In [23]:
import json
import plotly.graph_objects as go
import pandas as pd

edge_list = pd.read_csv("filtered_edge_list.csv")
with open("data/groups.json") as f:
    groups = json.load(f)["783"]

# invert groups -> id_to_group
id_to_group = {}
for grp, ids in groups.items():
    for i in ids:
        id_to_group[str(i)] = grp

def id_to_grp(i):
    return id_to_group.get(str(i), "Other")

edge_list['pre_group'] = edge_list['pre_root_id'].apply(id_to_grp)
edge_list['post_group'] = edge_list['post_root_id'].apply(id_to_grp)

edge_list.groupby(['pre_group', 'post_group'])['syn_count'].sum().reset_index().to_csv("grouped_edge_list.csv", index=False)

## 1.5 Construct the Sankey

The library used to visualize the Sankey, `plotly`, uses lists of sources, targets, and values. The sources, targets, and values only take integers, which correspond to the indices of the `labels`. `name_to_idx` maps the name of the group to its index in `labels`.

Essentially, there's some boilerplate needed to represent the data in a way that `plotly` understands and can use to build the diagram.

In [24]:
import pandas as pd
import json

aggregated = pd.read_csv("grouped_edge_list.csv")

# Build global labels
name_to_idx = {}
labels = []

source = []
target = []
value = []

for group in aggregated['pre_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for group in aggregated['post_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for _, row in aggregated.iterrows():
    source.append(name_to_idx[row['pre_group']])
    target.append(name_to_idx[row['post_group']])
    value.append(row['syn_count'])


name_to_color = { # Key by group name for easier identification
    "Ir94e": {"line": "purple", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (L)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (R)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T2 (L)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "GNG.SLP.T2 (R)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Earmuff": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley Glu interneuron": {"line": "red", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley ACh interneuron": {"line": "red", "arrow": "rgba(0, 255, 0, 0.5)"},
    "OviDN": {"line": "blue", "arrow": "rgba(0, 255, 0, 0.5)"},
}

node_colors = [name_to_color.get(label, {"line": "grey"})['line'] for label in labels]
arrow_colors = [name_to_color.get(labels[src], {"arrow": "rgba(128, 128, 128, 0.5)"})['arrow'] for src in source]

# plot Sankey
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors
    ),
    link=dict(
        arrowlen=15,
        source=source,
        target=target,
        value=value,
        color=arrow_colors
    )
)])
fig.update_layout(title_text="Ir94e → interneuron groups → OviDN (aggregated)", font_size=10)
fig.show()

# Part 2: Comparison against v630 data

The original paper used FAFB v630, but since then FlyWire has released v783, which includes more annotations, revised edge weights, etc. The above analysis used v783, so we want see if there are any substantial differences from the published results.

## 2.1 Download edge list from materialization v630

FlyWire doesn't offer v630 datasets for download anymore, so we will have to query the live database using `fafbseg`. The original paper limited the pathway length to three hops, so we can reconstruct that by doing the following:

* Get all immediate partners of Ir94e (hop 1: Ir94e -> 1° interneuron)
* Get the partners of the 1° interneurons (hop 2: 1° interneuron -> 2° interneurons).
* Get the partners of OviDNs (hop 3: 2° interneurons -> OviDNs). Because all edges are guaranteed to be connected to OviDNs, this minimizes the search space much more than if we were to again find the neurons downstream of the 2° interneurons and then perform a pathway analysis.

I included both pre- and post-synaptic partners when performing these searches, but it doesn't matter much since we're just looking at the forward direction of the circuit. The shortest paths analysis will use the directed edges to reach OviDNs regardless. The only advantage of forcing forward directionality is that you'd be working with less data, and the search on CAVE might be a bit faster.

In [ ]:
from fafbseg import flywire
import pandas as pd
import json

neurons = json.load(open("data/groups.json"))

first_hop: pd.DataFrame = flywire.synapses.get_connectivity(neurons["630"]["Ir94e"], filtered=True, materialization=630)
first_hop = first_hop[first_hop['weight'] >= 5]
print(f"{len(first_hop)} edges in the first hop.")

second_hop: pd.DataFrame = flywire.synapses.get_connectivity(first_hop['post'].unique().tolist(), upstream=True,filtered=True, materialization=630)
second_hop = second_hop[second_hop['weight'] >= 5]
print(f"{len(second_hop)} edges in the second hop.")

print(len(second_hop))  # raw row count — compare this number directly against the pre-fix run

intermediate_ids = set(first_hop['post'].unique())
print((second_hop['post'].isin(intermediate_ids)).sum(), "rows where intermediate is still on the POST side")

third_hop: pd.DataFrame = flywire.synapses.get_connectivity(neurons["630"]["OviDN"], downstream=True, filtered=True, materialization=630)
third_hop = third_hop[third_hop['weight'] >= 5]
print(f"{len(third_hop)} edges in the third hop.")

edge_list = pd.concat([first_hop, second_hop, third_hop], ignore_index=True).drop_duplicates()
edge_list.to_csv("630_edge_list.csv", index=False)

303 edges in the first hop.


Fetching connectivity:   0%|          | 0/3 [00:00<?, ?it/s]

5322 edges in the second hop.
5322
2645 rows where intermediate is still on the POST side
235 edges in the third hop.


## 2.2 Find the shortest paths

In [42]:
import pandas as pd
import networkx as nx

edge_list = pd.read_csv("630_edge_list.csv")
G = nx.from_pandas_edgelist(edge_list, source="pre", target="post", create_using=nx.DiGraph)

path_edges = []

for ir94e in neurons["630"]["Ir94e"]:
    for ovidn in neurons["630"]["OviDN"]:
        print(f"finding pathways between {ir94e} -> {ovidn}...", end="")
        paths: Generator[list, None, None] = nx.algorithms.all_shortest_paths(G, ir94e, ovidn)
        print("done")
        for path in paths:
            if len(path) > 4: continue # probably redundant but whatever
            path_edges.extend(zip(path, path[1:]))

path_edges_df = pd.DataFrame(path_edges, columns=["pre", "post"]).drop_duplicates()
print(f"unique path edges discovered: {len(path_edges_df.drop_duplicates())} ({len(path_edges_df)} total)")

finding pathways between 720575940621375231 -> 720575940632512156...done
finding pathways between 720575940621375231 -> 720575940640872923...done
finding pathways between 720575940621375231 -> 720575940621257340...done
finding pathways between 720575940621375231 -> 720575940613316783...done
finding pathways between 720575940621375231 -> 720575940642312136...done
finding pathways between 720575940638218173 -> 720575940632512156...done
finding pathways between 720575940638218173 -> 720575940640872923...done
finding pathways between 720575940638218173 -> 720575940621257340...done
finding pathways between 720575940638218173 -> 720575940613316783...done
finding pathways between 720575940638218173 -> 720575940642312136...done
finding pathways between 720575940626016017 -> 720575940632512156...done
finding pathways between 720575940626016017 -> 720575940640872923...done
finding pathways between 720575940626016017 -> 720575940621257340...done
finding pathways between 720575940626016017 -> 7205

## 2.3 Filter the larger edge list

In [ ]:
circuit_edges = edge_list.merge(
    path_edges_df,
    on=["pre", "post"],
    how="inner",
)
circuit_edges.to_csv("630_filtered_edge_list.csv", index=False)
print(circuit_edges.shape)  # should be close to 192, same order of magnitude as v783's 302

(192, 3)


## 2.4 Pool edges to the group level

In [44]:
import json
import plotly.graph_objects as go
import pandas as pd

edge_list = pd.read_csv("630_filtered_edge_list.csv")
with open("data/groups.json") as f:
    groups = json.load(f)["630"]

# invert groups -> id_to_group
id_to_group = {}
for grp, ids in groups.items():
    for i in ids:
        id_to_group[str(i)] = grp

def id_to_grp(i):
    return id_to_group.get(str(i), "Other")

edge_list['pre_group'] = edge_list['pre'].apply(id_to_grp)
edge_list['post_group'] = edge_list['post'].apply(id_to_grp)

edge_list.groupby(['pre_group', 'post_group'])['weight'].sum().reset_index().to_csv("630_grouped_edge_list.csv", index=False)

## 2.5 Construct the Sankey

In [45]:
import pandas as pd

aggregated = pd.read_csv("630_grouped_edge_list.csv")
# aggregated = aggregated[(aggregated['pre_group'] != "Other") & (aggregated['post_group'] != "Other")]

# Build global labels
name_to_idx = {}
labels = []

source = []
target = []
value = []

for group in aggregated['pre_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for group in aggregated['post_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for _, row in aggregated.iterrows():
    source.append(name_to_idx[row['pre_group']])
    target.append(name_to_idx[row['post_group']])
    value.append(row['weight'])


name_to_color = { # Key by group name for easier identification
    "Ir94e": {"line": "purple", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (L)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (R)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T2 (L)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "GNG.SLP.T2 (R)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Earmuff": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley Glu interneuron": {"line": "red", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley ACh interneuron": {"line": "red", "arrow": "rgba(0, 255, 0, 0.5)"},
    "OviDN": {"line": "blue", "arrow": "rgba(0, 255, 0, 0.5)"},
}

node_colors = [name_to_color.get(label, {"line": "grey"})['line'] for label in labels]
arrow_colors = [name_to_color.get(labels[src], {"arrow": "rgba(128, 128, 128, 0.5)"})['arrow'] for src in source]

# plot Sankey
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors
    ),
    link=dict(
        arrowlen=15,
        source=source,
        target=target,
        value=value,
        color=arrow_colors
    )
)])
fig.update_layout(title_text="Ir94e → interneuron groups → OviDN (aggregated)", font_size=10)
fig.show()

## Initial conclusions prior to characterizations of "Other" neurons

Synapses between known groups using v630 matches exactly what we see in the paper, which is great! v783 only adds more snyapses, which is also a good sign.

In both snapshots, Ir94e synapses onto itself (which makes sense biologically). It likely didn't show up in the paper for simplicity.

The big question lies in the "Other" neurons. They weren't mentioned in the paper yet play a big role in the v630 Sankey. They're even more prominent in the v783 circuit. Their characterization will tell us their significance in the function of the circuit.